<a href="https://colab.research.google.com/github/Gayatri8-sys/Machine-Learning/blob/main/NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install tensorflow==2.12.0 numpy==1.23.5 pandas scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import tensorflow as tf

print("NumPy version:", np.__version__)
print("TensorFlow version:", tf.__version__)


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier

In [ ]:
# Load dataset
df = pd.read_csv('/content/Alphabets_data.csv')

df

In [ ]:
df.describe()

In [ ]:
df['letter'].nunique()

In [ ]:
df.isnull().sum()

In [ ]:
col=df.columns
data1=df.copy()
num_col=data1.select_dtypes(include=['number']).columns
for c in num_col:
  q1=data1[c].quantile(0.25)
  q3=data1[c].quantile(0.75)
  iqr=q3-q1
  lower_bound=q1-1.5*iqr
  upper_bound=q3+1.5*iqr
  data1=data1[(data1[c]>=lower_bound) & (data1[c]<=upper_bound)]

In [ ]:
from sklearn.model_selection import train_test_split
X=data1.drop(columns=['letter'])
Y=data1['letter']

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
Y_encoded=le.fit_transform(Y)

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X_scaled,Y_encoded,test_size=0.2,random_state=42)

In [ ]:
model=Sequential() # ANN model building

# input layer + hidden layer with 64 neurons
model.add(Dense(units=64, activation='relu', input_dim=X_train.shape[1]))

# model was overfitting , so dropuout 30% neurons randomly
from keras.layers import Dropout
model.add(Dropout(0.1))

# output layer with 26 neurons bcoz 26 classes
model.add(Dense(units=26, activation='softmax'))

Softmax activation is specifically designed for multi-class classification problems where you have more than two classes ? like your Alphabet dataset with 26 letters (A to Z).



In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# fit model to train data, train the model

model.fit(X_train, Y_train, epochs=20, batch_size=32, validation_split=0.1)

Optimizer controls how the model updates its weights based on the loss function during training.


Adam stands for Adaptive Moment Estimation.



One epoch means the model has seen all the training data once.

batch size-Number of samples processed before the model updates weights.

In [ ]:
# evaluate model

loss, accuracy = model.evaluate(X_test, Y_test)
print(f"Test Accuracy: {accuracy:.2f}")

In [ ]:
history = model.fit(X_train, Y_train,
                    epochs=20,
                    batch_size=32,
                    validation_split=0.1,
                    verbose=1)
# Plot accuracy over epochs
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.show()

Hyperparameter Tuning

In [ ]:
def create_model(units=64, optimizer='adam', drop=0.1):
    model = Sequential()
    model.add(Dense(units, input_dim=X_train.shape[1], activation='relu'))
    model.add(Dropout(drop))
    model.add(Dense(units=26, activation='softmax')) # Changed activation to softmax for multi-class classification
    model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy']) # Changed loss to sparse_categorical_crossentropy
    return model

In [ ]:
model = KerasClassifier(build_fn=create_model, verbose=0)

In [ ]:
param_grid = {
    'batch_size': [32, 64],
    'epochs': [10, 20],
    'units': [32, 64, 128], # Changed to pass directly to create_model
    'optimizer': ['adam', 'sgd'], # Changed to pass directly to create_model
    'drop': [0.1, 0.2, 0.3] # Changed to pass directly to create_model
}

grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, scoring='accuracy') # Added scoring='accuracy'
grid_result = grid.fit(X_train, Y_train)

In [ ]:
best_params = grid_result.best_params_
print(best_params)

In [ ]:
def create_model(units=64, optimizer='adam', drop=0.1):
    model = Sequential()
    model.add(Dense(units, input_dim=X_train.shape[1], activation='relu'))  # assuming 26 features
    model.add(Dropout(drop))
    model.add(Dense(1, activation='softmax'))
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model


In [ ]:
final_model = create_model(
    units=best_params['units'],
    optimizer=best_params['optimizer'],
    drop=best_params['drop']
)

In [ ]:
final_model.fit(X_train, Y_train,
                batch_size=best_params['batch_size'],
                epochs=best_params['epochs'],
                verbose=1)

In [ ]:
Y_pred=final_model.predict(X_test)

In [ ]:
loss, accuracy = final_model.evaluate(X_test, Y_test)
print(f"Test Accuracy: {accuracy:.2f}")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# Convert predictions to class labels
Y_pred_classes = np.argmax(Y_pred, axis=1)

print("Classification Report:\n", classification_report(Y_test, Y_pred_classes))
print("Accuracy:", accuracy_score(Y_test, Y_pred_classes))
print("Precision:", precision_score(Y_test, Y_pred_classes, average='weighted'))
print("Recall:", recall_score(Y_test, Y_pred_classes, average='weighted'))
print("F1 Score:", f1_score(Y_test, Y_pred_classes, average='weighted'))

?	Discuss the performance differences between the model with default hyperparameters and the tuned model, emphasizing the effects of hyperparameter tuning



The default model, built with standard hyperparameters, provided a decent baseline but showed limited performance, likely due to underfitting or suboptimal settings. After applying hyperparameter tuning using GridSearchCV, the model?s accuracy and F1-score significantly improved. Tuning parameters like the number of neurons, activation functions, optimizers, and batch sizes allowed the model to learn better patterns and generalize more effectively. Overall, hyperparameter tuning led to a more robust and accurate classifier, highlighting its importance in optimizing neural network performance.



